# Implementing the betaprime kernel

The betaprime kernel is defined in the following way: 

$$
\mathcal{K}(A, B) = \Gamma_M(2 \alpha) \left[\frac{\mathrm{det}(A) \mathrm{det}(B}{\mathrm{det}(A + B)}\right]^{\alpha}    \quad \text{for } \alpha > n- 1
$$


In [ ]:
import numpy as np 
import matplotlib.pyplot as plt

def pairwise_betaprime(X, Y, alpha = None):   
    """
    Computes the pairwise beta prime kernel between two matrices X and Y.
    """
    # Check if X and Y have the same shape
    if alpha is None:
        alpha = X.shape[0] + 1
    if X.shape != Y.shape:
        raise ValueError("X and Y must have the same shape")
    
    K = (np.linalg.det(X) * np.linalg.det(Y) / np.linalg.det(X + Y))**(alpha)

    return K

def betaprime_kernel(covariances, alpha = None):

    """
    Computes the beta prime kernel between covariance matrices.
    Parameters
    ----------
    covariances : array-like, shape (n_samples, n_features, n_features)
        Covariance matrices to compute the kernel for.
    alpha : int, optional
        The number of features in the covariance matrices. Default is the number of features in the first covariance matrix.
    Returns
    -------
    K : array-like, shape (n_samples, n_samples)
        The beta prime kernel matrix.
    """
    if alpha is None:
        alpha = covariances.shape[1] + 1
    
    n_samples = covariances.shape[0]
    K = np.zeros((n_samples, n_samples))
    for i in range(n_samples):
        K[i, i] = pairwise_betaprime(covariances[i], covariances[i], alpha)
        for j in range(i+1, n_samples):
            K[i, j] = pairwise_betaprime(covariances[i], covariances[j], alpha)
            K[j, i] = K[i, j]
    return K

def plot_kernel_matrix(K):
    """
    Plots the kernel matrix.
    Parameters
    ----------
    K : array-like, shape (n_samples, n_samples)
        The kernel matrix to plot.
    """
    plt.imshow(K, cmap='hot', interpolation='nearest')
    plt.colorbar()
    plt.title('Beta Prime Kernel Matrix')
    plt.show()




In [ ]:
# Load the data from A01T.gdf
import mne 
import scipy 
from mne import Epochs, pick_types, events_from_annotations
from mne.datasets import sample
from scipy.io import loadmat

# Load the BNCI dataset already downloaded

data_path = "/Users/rishabhkumar/mne_data/MNE-bnci-data/database/data-sets/001-2014/"
mat = loadmat(data_path + 'A01T.mat', struct_as_record=False, squeeze_me=True)



{'__header__': b'MATLAB 5.0 MAT-file, Platform: GLNXA64, Created on: Thu Aug  7 13:07:03 2014',
 '__version__': '1.0',
 '__globals__': [],
 'data': array([<scipy.io.matlab._mio5_params.mat_struct object at 0x17aa31a80>,
       dtype=object)}

In [86]:
# Preprocess the data
from scipy.io.matlab import mat_struct

def _check_keys(d: dict) -> dict:
    """
    Find entries in the dict that are mat_struct and convert them.
    """
    for key, value in d.items():
        if isinstance(value, mat_struct):
            d[key] = _todict(value)
        elif isinstance(value, np.ndarray) and value.dtype == object:
            d[key] = _tolist(value)
    return d

def _todict(matobj: mat_struct) -> dict:
    """
    Recursively convert a mat_struct into a dict.
    """
    d = {}
    for field in matobj._fieldnames:           # mat_struct stores field names here
        elem = getattr(matobj, field)
        if isinstance(elem, mat_struct):
            d[field] = _todict(elem)
        elif isinstance(elem, np.ndarray) and elem.dtype == object:
            d[field] = _tolist(elem)
        else:
            d[field] = elem
    return d

def _tolist(ndarray: np.ndarray) -> list:
    """
    Recursively convert object arrays: mat_struct → dict, arrays → lists.
    """
    result = []
    for item in ndarray:
        if isinstance(item, mat_struct):
            result.append(_todict(item))
        elif isinstance(item, np.ndarray):
            result.append(_tolist(item))
        else:
            result.append(item)
    return result

In [105]:
def loadBCIdata(data_path, filename = 'A01T.mat'):
    """
    Load the BCI dataset.
    Parameters
    ----------
    data_path : str
        Path to the BCI dataset.
    Returns
    -------
    data : dict
        Dictionary containing the data and labels.
    """
    # Load the data from the .mat file
    mat = loadmat(data_path + filename, struct_as_record=False, squeeze_me=True)
    # Extract the data and labels
    mat = _check_keys(mat)
    new_data = mat['data']
    # labels = mat['labels']
    return new_data

def preprocess_data(data, labels):
    """
    Preprocess the data.
    Parameters
    ----------
    data : array-like, shape (n_samples, n_features)
        The data to preprocess.
    labels : array-like, shape (n_samples,)
        The labels for the data.
    Returns
    -------
    data : array-like, shape (n_samples, n_features)
        The preprocessed data.
    labels : array-like, shape (n_samples,)
        The preprocessed labels.
    """
    # Normalize the data
    data = (data - np.mean(data)) / np.std(data)
    
    # Reshape the data
    data = data.reshape(data.shape[0], -1)
    
    return data, labels


In [120]:
data = loadBCIdata(data_path)
data[3]
# X = np.stack([data['X'][i] for i in range(len(data['X']))])

{'X': array([[  0.34179688,   0.24414062,  -3.22265625, ...,  10.25390625,
          20.5078125 ,   5.859375  ],
        [ -6.34765625,  -7.95898438, -10.49804688, ...,   2.44140625,
           7.8125    ,  -4.8828125 ],
        [ -1.80664062,  -7.17773438,  -8.15429688, ...,   6.34765625,
          13.671875  ,  -0.48828125],
        ...,
        [  0.29296875,   0.04882812,  -3.07617188, ...,  -2.44140625,
          45.8984375 ,  -4.39453125],
        [ -6.34765625,  -5.12695312,  -9.71679688, ...,  -4.8828125 ,
          38.57421875,  -4.39453125],
        [ -9.71679688,  -7.17773438, -13.96484375, ...,  -5.37109375,
          39.55078125, -10.7421875 ]]),
 'trial': array([  251,  2254,  4172,  6124,  8132, 10243, 12160, 14210, 16141,
        18139, 20045, 21940, 23912, 25856, 27823, 29943, 31951, 34017,
        36053, 38119, 40189, 42269, 44181, 46212, 48165, 50126, 52230,
        54234, 56209, 58235, 60306, 62413, 64505, 66596, 68639, 70703,
        72723, 74695, 76658, 78583, 806

In [89]:
# 1. Load .mat
mat = loadmat(data_path + 'A02T.mat', struct_as_record=False, squeeze_me=True)

# 2. Clean up top-level struct keys
mat = _check_keys(mat)

# 3. Now, for example, your data field:
#    If originally mat['data'] was a 1×9 struct array,
#    after conversion you have:
data_list = mat['data']             # a Python list of 9 dicts
first_trial = data_list[0]          # dict for the first trial
# eeg_array = first_trial['x']        # assuming the field storing signals is named 'x'

# 4. Labels likewise:
# labels = mat['classlabel']    

In [ ]:
 # number of trials

X = np.stack([getattr(trial, 'X') for trial in mat['data']])


AttributeError: 'dict' object has no attribute 'X'

NameError: name 'trial0' is not defined